## Training-Serving Skew

Training-serving skew is the divergence between the data distribution or feature preprocessing
the model was trained on and the data it receives at inference time in production.
It is one of the most common causes of unexplained model performance degradation in live systems.

## How it actually works

A model learns a mapping from features to outputs during training.
At serving time, it applies that mapping to new inputs.
If the inputs at serving time look statistically different from the training inputs,
the model is making predictions outside its learned distribution.
It has no way to signal this — it just produces an output.

**The three main causes:**

1. **Preprocessing inconsistency:**
The most common. During training, you compute a feature like
`log(invoice_amount)` with a scaler fitted on the training set.
At serving time, someone rewrites the preprocessing code slightly.
The scaler is refitted on a different dataset, or the log transform is omitted.
The model receives different numeric values than it was trained on.
Performance degrades. Nobody knows why. The code looks correct.

2. **Feature availability skew:**
Training used a feature computed from historical data — e.g. a 90-day rolling average.
At serving time for a new record, that 90-day history does not exist yet.
The serving pipeline either fills in zero, uses a shorter window, or drops the feature.
The model never saw that substitution during training.

3. **Data distribution shift:**
Training data was from 2022-2023. It is now 2025.
Customer behaviour, transaction patterns, and external factors have changed.
The model's learned boundaries no longer reflect the real world.
This is distinct from preprocessing skew — the code is correct, but the world has changed.

**How to detect it:**
- Compare the statistical distribution of each feature in training data vs live inference requests.
  Mean, standard deviation, min/max, histogram shape.
- Tools: evidently, deepchecks, Whylogs, custom monitoring pipelines.
- Log raw incoming features at inference time alongside predictions.
  You cannot debug skew if you do not store what the model actually received.

**How to prevent preprocessing skew specifically:**
The gold standard is a single preprocessing function that runs identically at training time and serving time.
In scikit-learn: fit a Pipeline object during training, serialise it with joblib, load and call it at serving time.
Same code path. No drift possible.
In more complex systems: publish the preprocessing logic as a shared library or a feature store.
Both training and serving read from the same source.

## Where it breaks or gets dangerous

**1. Silent degradation — the most dangerous pattern:**
Training-serving skew rarely throws an error. It produces a prediction.
That prediction may be slightly off, consistently off in one direction, or systematically wrong for a segment.
Without ground truth labels flowing back and without feature monitoring,
you can run a skewed model in production for months before anyone notices.

**2. Data leakage in training — a specific form of skew in reverse:**
During training, you accidentally use a feature that is only available after the outcome is known.
For example: using the settlement status of an invoice to predict payment delay.
The model trains with very high accuracy. In production, that feature is not yet available at decision time.
The model collapses. The training metrics were meaningless.
This is temporal skew — the feature timeline at training time differs from the feature timeline at serving time.

**3. Serving pipeline written independently from training code:**
Training done by data scientist in Python. Serving pipeline written by backend engineer in Java.
Both are implementing the same specification. Both are slightly wrong in different ways.
No shared code. No automated comparison. Skew builds up silently.

**4. Schema changes upstream:**
Source system changes a field name, data type, or encoding.
Training data was captured before the change. Serving pipeline receives data after the change.
The model receives an unexpected null or an incorrectly encoded category.
No warning — the pipeline does not know the schema changed.

**5. Skew in enterprise AI is a compliance risk:**
If a model is used in a regulated context — credit, fraud detection, financial reporting —
and it is producing predictions on data that diverges from its validated training distribution,
you may be in violation of model governance requirements.
Model validation documents the training distribution.
Serving on a different distribution means the validation no longer applies.

In [ ]:
# Training-Serving Skew — demonstration

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Simulate a training dataset: invoice features
# Feature 0: log(invoice_amount), Feature 1: days_since_last_payment
X_train = np.column_stack([
    np.log(np.random.uniform(100, 50000, 1000)),  # log-transformed
    np.random.uniform(0, 90, 1000)
])
y_train = (X_train[:, 0] > np.median(X_train[:, 0])).astype(int)

X_test = np.column_stack([
    np.log(np.random.uniform(100, 50000, 200)),
    np.random.uniform(0, 90, 200)
])
y_test = (X_test[:, 0] > np.median(X_train[:, 0])).astype(int)

# Correct approach: Pipeline serialises scaler + model together
correct_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
correct_pipeline.fit(X_train, y_train)
correct_preds = correct_pipeline.predict(X_test)
correct_acc = accuracy_score(y_test, correct_preds)

print("=" * 60)
print("CORRECT: single Pipeline — scaler fitted once at training time")
print(f"Test accuracy: {correct_acc:.3f}")

# Skewed approach: serving team re-fits scaler on test data
# (simulates a different preprocessing path at serving time)
skewed_scaler = StandardScaler()
X_test_skewed = skewed_scaler.fit_transform(X_test)  # fitted on test data, not training data

# Use only the trained LogisticRegression (bypassing the pipeline's scaler)
skewed_model = correct_pipeline.named_steps["model"]
# But we need to apply the training scaler first for the comparison to be meaningful
# Here the bug is: test data was scaled with test-data statistics instead of training statistics
skewed_preds = skewed_model.predict(X_test_skewed)
skewed_acc = accuracy_score(y_test, skewed_preds)

print()
print("=" * 60)
print("SKEWED: serving pipeline re-fits scaler on inference data")
print("(Different mean/std used — model receives values outside training distribution)")
print(f"Test accuracy: {skewed_acc:.3f}")

print()
print("=" * 60)
print("Feature distribution comparison (training vs serving):")
train_mean = correct_pipeline.named_steps["scaler"].mean_
train_std = np.sqrt(correct_pipeline.named_steps["scaler"].var_)
serving_mean = skewed_scaler.mean_
serving_std = np.sqrt(skewed_scaler.var_)

print(f"Feature 0 mean — training: {train_mean[0]:.3f}, serving: {serving_mean[0]:.3f}")
print(f"Feature 0 std  — training: {train_std[0]:.3f}, serving: {serving_std[0]:.3f}")
print(f"Feature 1 mean — training: {train_mean[1]:.3f}, serving: {serving_mean[1]:.3f}")
print()
print("Fix: serialise the fitted Pipeline. Load and call it at serving time.")
print("Same scaler. Same statistics. No skew possible.")

## My D365 analogy

Training-serving skew is the ML equivalent of a posting profile mismatch between environments.

In D365, a posting profile defines how transactions are mapped to ledger accounts.
If your UAT environment has one posting profile configuration
and production has a different one — same transaction,
different ledger posting, different financial result.
Both environments run. No error is thrown.
The difference only surfaces when you reconcile the subledger to the general ledger
and the numbers don't match.

Training-serving skew works the same way.
Training pipeline uses one preprocessing configuration.
Serving pipeline uses a slightly different one.
Both pipelines run without error.
The difference only surfaces when you compare predictions to actual outcomes
and the accuracy is lower than expected.

The fix in D365: a single posting profile definition, shared between environments,
deployed through the release pipeline.
You never configure posting profiles manually in production.

The fix in ML: a single preprocessing artifact, serialised at training time,
deployed to serving. You never refit transformations in production.

One configuration, one source of truth, deployed — not replicated — to every environment.
Same principle. Different stack.

## LinkedIn post idea

In D365, a posting profile misconfiguration between UAT and production
is one of the first things you check when subledger reconciliation fails.
Same transaction. Different environments. Different ledger entries.
No error. Just wrong numbers.

Training-serving skew in ML is exactly the same failure mode.

You train a model with a scaler fitted on training data.
The serving team rebuilds the preprocessing step independently.
Slightly different statistics. Same intent. Different execution.
The model runs. No exceptions. Lower accuracy. Nobody knows why.

The fix is the same principle I've applied in ERP implementations for 14 years:
one source of truth, deployed — not replicated — across environments.

In D365: a posting profile deployed via release pipeline.
In ML: a serialised preprocessing pipeline loaded from a model registry.

Enterprise domain knowledge does not become irrelevant in AI engineering.
It just puts a different name on problems you have already solved.